In [73]:
import pandas as pd

for_attack_gen_data = 'C:\\AIGuard\\for_attack_gen'
for_defense_gen_data = 'C:\\AIGuard\\for_defense_gen'

# 해커가 사용하는 모든 공격 기법
techniques = pd.read_excel(f"{for_attack_gen_data}\\ATT&CK_enterprise-attack-v19.2-techniques.xlsx")

# 해커 공격 시 어떻게 해야할지 대응 정보 - description
mitigations = pd.read_excel(f"{for_defense_gen_data}\\ATT&CK_enterprise-attack-v19.2-mitigations.xlsx")

# 공격 패턴의 개연성 데이터
relationships = pd.read_excel(f"{for_attack_gen_data}\\ATT&CK_enterprise-attack-v19.2-relationships.xlsx")

techniques = techniques[['ID','name', 'description', 'tactics', 'platforms']]
mitigations = mitigations[['ID','name', 'description', 'relationship citations']]
relationships = relationships[['source ID', 'source name', 'source type', 'mapping type',
                               'target ID', 'target name', 'target type', 'mapping description']]

techniques = techniques.rename(columns={'ID' : 'target ID', 
                                        'name' : 'tech_name',
                                        'description' : 'tech_description'})

mitigations = mitigations.rename(columns={'ID' : 'source ID', 
                                        'name' : 'miti_name',
                                        'description' : 'miti_description'})

# # 2. '공격 기술 ID'와 '방어책 ID' 연결하기
security_map = relationships.merge(techniques, on="target ID")
security_map = security_map.merge(mitigations, on="source ID")

# security_map.shape

In [74]:
detectionstrategies = pd.read_excel(f"{for_defense_gen_data}\\ATT&CK_enterprise-attack-v19.2-detectionstrategies.xlsx")
analytics = pd.read_excel(f"{for_defense_gen_data}\\ATT&CK_enterprise-attack-v19.2-analytics.xlsx")

analytics['detect_ID'] = analytics['url'].str.split('/').str[-1].str.split('#').str[0]

analytics = analytics[['ID','name','url','description','detect_ID']]
detectionstrategies = detectionstrategies[['ID', 'name']]

analytics = analytics.rename(columns={'ID' : 'anal_ID', 
                                        'name' : 'anal_name',
                                        'description' : 'anal_description'})

detectionstrategies = detectionstrategies.rename(columns={'ID' : 'detect_ID', 
                                        'name' : 'detect_name',
                                        'description' : 'detect_description'})

solution_info = analytics.merge(detectionstrategies, on = 'detect_ID')
solution_info.head()

,anal_ID,anal_name,url,anal_description,detect_ID,detect_name
0,AN0001,Analytic 0001,https://attack.mitre.org/detectionstrategies/D...,Detects access attempts to cloud instance meta...,DET0001,Detect Access to Cloud Instance Metadata API (...
1,AN0002,Analytic 0002,https://attack.mitre.org/detectionstrategies/D...,"Detects non-standard processes (e.g., PowerShe...",DET0002,Behavioral Detection of Publish/Subscribe Prot...
2,AN0003,Analytic 0003,https://attack.mitre.org/detectionstrategies/D...,"Detects CLI tools (e.g., mosquitto_pub, nc, py...",DET0002,Behavioral Detection of Publish/Subscribe Prot...
3,AN0004,Analytic 0004,https://attack.mitre.org/detectionstrategies/D...,"Detects osascript, curl, or custom binaries in...",DET0002,Behavioral Detection of Publish/Subscribe Prot...
4,AN0005,Analytic 0005,https://attack.mitre.org/detectionstrategies/D...,"Detects pub/sub traffic over unusual ports, hi...",DET0002,Behavioral Detection of Publish/Subscribe Prot...


In [75]:
import requests
from bs4 import BeautifulSoup

def find_target_id(target_url):
    response = requests.get(target_url, headers={'User-Agent': 'Mozilla/5.0'})
    soup = BeautifulSoup(response.text, 'html.parser')

    main_content = soup.find('h4').find('a').get_text().strip().split('|')[1].strip()
    return main_content

In [76]:
from concurrent.futures import ThreadPoolExecutor
import requests
from bs4 import BeautifulSoup

# 동시에 20개씩 병렬 처리 (속도 10~20배 향상)
urls = solution_info['url'].tolist()
with ThreadPoolExecutor(max_workers=20) as executor:
    results = list(executor.map(find_target_id, urls))

solution_info['target ID'] = results

In [80]:
security_map = security_map.merge(solution_info, on='target ID')

In [83]:
security_map.to_csv('security_map.csv',index=False)